# Notebook 04 · Dataset E1 — Augmentación Moderada (+50 % Clase Fake)

Este notebook construye el escenario experimental **E1** para el fine-tuning
de los modelos *RoBERTa-BNE* y *MarIA* sobre el corpus FakeDeS.
E1 incorpora textos sintéticos generados en HU09 al split de entrenamiento,
incrementando en aproximadamente un 50 % el número de ejemplos de la clase
falsa. Los splits de validación y prueba son idénticos a los del escenario E0.

| Artefacto de salida | Ruta |
|---|---|
| Split de entrenamiento augmentado | `data/processed/E1/train.csv` |
| Split de validación | `data/processed/E1/val.csv` |
| Split de prueba | `data/processed/E1/test.csv` |
| Ficha de trazabilidad | `data/processed/E1/metadata_E1.json` |

## § 0 · Dependencias

Importaciones necesarias para la construcción, serialización y verificación
del dataset E1.

In [3]:
import warnings
from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime, timezone
warnings.filterwarnings('ignore')

## § 1 · Configuración y Carga de Datos

E1 es el escenario **augmentado** del experimento: extiende el split de
entrenamiento de E0 con textos sintéticos de clase falsa generados mediante
prompting few-shot con GPT-4o-mini (HU09) y filtrados por similitud semántica
con *paraphrase-multilingual-mpnet-base-v2* (HU10). Los splits de validación
y prueba permanecen inalterados respecto a E0, garantizando la comparabilidad
de las métricas de clasificación entre escenarios (HU16).

In [5]:
ROOT_DIR      = Path().resolve().parent
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
E1_DIR        = PROCESSED_DIR / 'E1'
E1_DIR.mkdir(parents=True, exist_ok=True)

# Carga de splits originales y textos sintéticos
df_train = pd.read_csv(PROCESSED_DIR / 'train.csv',               encoding='utf-8')
df_val   = pd.read_csv(PROCESSED_DIR / 'val.csv',                 encoding='utf-8')
df_test  = pd.read_csv(PROCESSED_DIR / 'test.csv',                encoding='utf-8')
df_sint  = pd.read_csv(PROCESSED_DIR / 'augmented_synthetic.csv', encoding='utf-8')

for nombre, df in [('train', df_train), ('val', df_val),
                   ('test', df_test), ('augmented_synthetic', df_sint)]:
    print(f'{nombre:<25}  shape={df.shape}')

print(f'\nRegistros sintéticos disponibles: {len(df_sint)}')

train                      shape=(772, 7)
val                        shape=(97, 7)
test                       shape=(97, 7)
augmented_synthetic        shape=(185, 8)

Registros sintéticos disponibles: 185


## § 2 · Construcción del Dataset E1

Únicamente el split de entrenamiento incorpora los textos sintéticos generados
en HU09: se concatena el subconjunto original de la clase falsa con los
registros sintéticos filtrados, obteniendo un conjunto ampliado con una
proporción de clase falsa aproximada del 50–60 %. Los splits de validación y
prueba se copian directamente desde los archivos producidos en el Notebook 01,
sin ninguna modificación, asegurando que la evaluación de los modelos se
realice en condiciones idénticas a las del escenario E0.

In [7]:
# Selección de columnas y copia defensiva
df_train_base = df_train[['Text', 'Category']].copy()
df_val_E1     = df_val[['Text', 'Category']].copy()
df_test_E1    = df_test[['Text', 'Category']].copy()

# Eliminación de filas con Text nulo o vacío en los tres splits
for df in (df_train_base, df_val_E1, df_test_E1):
    mask = df['Text'].isna() | (df['Text'].astype(str).str.strip() == '')
    df.drop(index=df.index[mask], inplace=True)
    df.reset_index(drop=True, inplace=True)

# Preparación del bloque sintético (Category ya es 1)
df_sinteticos = df_sint[['Text', 'Category']].copy()
df_sinteticos = df_sinteticos[~(df_sinteticos['Text'].isna() |
                                (df_sinteticos['Text'].astype(str).str.strip() == ''))]
df_sinteticos = df_sinteticos.reset_index(drop=True)

# Concatenación y mezcla del split de entrenamiento
n_sint = len(df_sinteticos)
df_train_E1 = (pd.concat([df_train_base, df_sinteticos], ignore_index=True)
                 .sample(frac=1, random_state=42)
                 .reset_index(drop=True))

# Tabla resumen
header = (f"{'Split':<6} {'Total':>7} {'Fake(1)':>8} "
          f"{'True(0)':>8} {'%Fake':>7} {'Sintéticos':>11}")
sep    = '-' * len(header)
print(header)
print(sep)

n_fake_orig = int((df_train_base['Category'] == 1).sum())

for nombre, df, n_s in [('train', df_train_E1, n_sint),
                         ('val',   df_val_E1,   0),
                         ('test',  df_test_E1,  0)]:
    n_total = len(df)
    n_fake  = int((df['Category'] == 1).sum())
    n_true  = int((df['Category'] == 0).sum())
    pct     = n_fake / n_total * 100 if n_total else 0.0
    print(f'{nombre:<6} {n_total:>7} {n_fake:>8} {n_true:>8} '
          f'{pct:>6.1f}% {n_s:>11}')

Split    Total  Fake(1)  True(0)   %Fake  Sintéticos
----------------------------------------------------
train      957      566      391   59.1%         185
val         97       48       49   49.5%           0
test        97       48       49   49.5%           0


## § 3 · Guardado de Archivos

Los tres splits de E1 se persisten en el directorio `data/processed/E1/`.
El hash MD5 de cada archivo se calcula sobre el contenido binario del fichero
ya escrito para garantizar la integridad de los artefactos en disco.

In [9]:
def _md5(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

splits = [
    ('train', df_train_E1, E1_DIR / 'train.csv'),
    ('val',   df_val_E1,   E1_DIR / 'val.csv'),
    ('test',  df_test_E1,  E1_DIR / 'test.csv'),
]

hashes = {}
for nombre, df, ruta in splits:
    df.to_csv(ruta, index=False, encoding='utf-8')
    hashes[nombre] = _md5(ruta)
    print(f'{nombre:<6}  registros={len(df):>6}  md5={hashes[nombre]}')

train   registros=   957  md5=d5949dfe6b3c2f42110b6865895fbebd
val     registros=    97  md5=2d22860c2f6cbfeccca34a75f1d356ff
test    registros=    97  md5=06697b127fd01b181ff5e50d87a5a2d3


## § 4 · Ficha de Trazabilidad E1

La ficha `metadata_E1.json` registra los parámetros de construcción del
escenario E1, las estadísticas descriptivas de cada split, los hashes MD5 de
los artefactos en disco y la tasa de augmentación efectiva. Este documento
permite reproducir el experimento E1 y comparar sus resultados con los del
escenario baseline E0.

In [11]:
n_fake_train_orig = int((df_train_base['Category'] == 1).sum())
n_fake_train_sint = int(n_sint)
n_fake_train_tot  = int((df_train_E1['Category'] == 1).sum())
n_true_train      = int((df_train_E1['Category'] == 0).sum())
prop_fake         = round(n_fake_train_tot / len(df_train_E1), 4) if len(df_train_E1) else 0.0
tasa_aug          = round(n_fake_train_sint / n_fake_train_orig, 4) if n_fake_train_orig else 0.0

metadata = {
    'escenario':               'E1',
    'descripcion':             'Corpus FakeDeS con augmentación sintética +50% clase fake. Dataset E1.',
    'fecha_construccion':      datetime.now(timezone.utc).isoformat(),
    'semilla_mezcla':          42,
    'fuente_train_base':       'data/processed/train.csv',
    'fuente_sinteticos':       'data/processed/augmented_synthetic.csv',
    'fuente_val':              'data/processed/val.csv',
    'fuente_test':             'data/processed/test.csv',
    'n_train':                 len(df_train_E1),
    'n_val':                   len(df_val_E1),
    'n_test':                  len(df_test_E1),
    'n_fake_train_original':   n_fake_train_orig,
    'n_fake_train_sintetico':  n_fake_train_sint,
    'n_fake_train_total':      n_fake_train_tot,
    'n_true_train':            n_true_train,
    'proporcion_fake_train':   prop_fake,
    'hash_md5_train':          hashes['train'],
    'hash_md5_val':            hashes['val'],
    'hash_md5_test':           hashes['test'],
    'columnas':                ['Text', 'Category'],
    'modelos_objetivo':        ['RoBERTa-BNE', 'MarIA'],
    'augmentacion_aplicada':   True,
    'tasa_augmentacion':       tasa_aug,
}

META_PATH = E1_DIR / 'metadata_E1.json'
with open(META_PATH, 'w', encoding='utf-8') as fh:
    json.dump(metadata, fh, ensure_ascii=False, indent=2)

print(json.dumps(metadata, ensure_ascii=False, indent=2))

{
  "escenario": "E1",
  "descripcion": "Corpus FakeDeS con augmentación sintética +50% clase fake. Dataset E1.",
  "fecha_construccion": "2026-06-04T19:43:37.383757+00:00",
  "semilla_mezcla": 42,
  "fuente_train_base": "data/processed/train.csv",
  "fuente_sinteticos": "data/processed/augmented_synthetic.csv",
  "fuente_val": "data/processed/val.csv",
  "fuente_test": "data/processed/test.csv",
  "n_train": 957,
  "n_val": 97,
  "n_test": 97,
  "n_fake_train_original": 381,
  "n_fake_train_sintetico": 185,
  "n_fake_train_total": 566,
  "n_true_train": 391,
  "proporcion_fake_train": 0.5914,
  "hash_md5_train": "d5949dfe6b3c2f42110b6865895fbebd",
  "hash_md5_val": "2d22860c2f6cbfeccca34a75f1d356ff",
  "hash_md5_test": "06697b127fd01b181ff5e50d87a5a2d3",
  "columnas": [
    "Text",
    "Category"
  ],
  "modelos_objetivo": [
    "RoBERTa-BNE",
    "MarIA"
  ],
  "augmentacion_aplicada": true,
  "tasa_augmentacion": 0.4856
}


## § 5 · Comparativa E0 vs E1

La siguiente tabla cuantifica el impacto de la augmentación sintética sobre
las estadísticas del split de entrenamiento. Los splits de validación y prueba
son idénticos en ambos escenarios y, por tanto, se omiten de la comparativa.

In [13]:
# Carga de la ficha de trazabilidad E0
with open(PROCESSED_DIR / 'E0' / 'metadata_E0.json', encoding='utf-8') as fh:
    meta_E0 = json.load(fh)

# Métricas comparadas
metricas = [
    ('n_train',               'Registros train',        meta_E0['n_train'],         metadata['n_train']),
    ('n_fake_train',          'Fake(1) train',           meta_E0['n_fake_train'],    metadata['n_fake_train_total']),
    ('n_true_train',          'True(0) train',           meta_E0['n_true_train'],    metadata['n_true_train']),
    ('proporcion_fake_train', '%Fake train',             meta_E0['proporcion_fake_train'],
                                                          metadata['proporcion_fake_train']),
]

header = f"{'Métrica':<26} {'E0':>10} {'E1':>10} {'Diferencia':>12}"
sep    = '-' * len(header)
print(header)
print(sep)
for _, etiqueta, v_e0, v_e1 in metricas:
    if isinstance(v_e0, float):
        diff_str = f'{v_e1 - v_e0:+.4f}'
        row = f'{etiqueta:<26} {v_e0:>10.4f} {v_e1:>10.4f} {diff_str:>12}'
    else:
        diff_str = f'{v_e1 - v_e0:+d}'
        row = f'{etiqueta:<26} {v_e0:>10} {v_e1:>10} {diff_str:>12}'
    print(row)

Métrica                            E0         E1   Diferencia
-------------------------------------------------------------
Registros train                   772        957         +185
Fake(1) train                     381        566         +185
True(0) train                     391        391           +0
%Fake train                    0.4935     0.5914      +0.0979


## § 6 · Verificación Final

Los tres archivos CSV de E1 se recargan desde disco para confirmar que los
hashes MD5 calculados coinciden con los registrados en `metadata_E1.json`.
La verificación garantiza la integridad del dataset E1 antes de su uso en
los experimentos de fine-tuning.

In [15]:
# Recarga desde disco
df_train_v = pd.read_csv(E1_DIR / 'train.csv', encoding='utf-8')
df_val_v   = pd.read_csv(E1_DIR / 'val.csv',   encoding='utf-8')
df_test_v  = pd.read_csv(E1_DIR / 'test.csv',  encoding='utf-8')

# Recalculo de hashes
hashes_v = {
    'train': _md5(E1_DIR / 'train.csv'),
    'val':   _md5(E1_DIR / 'val.csv'),
    'test':  _md5(E1_DIR / 'test.csv'),
}

# Tabla de verificación
header = f"{'Archivo':<20} {'Registros':>10} {'Hash MD5':<34} {'Estado':>6}"
sep    = '-' * len(header)
print(header)
print(sep)

filas = [
    ('E1/train.csv', df_train_v, 'train'),
    ('E1/val.csv',   df_val_v,   'val'),
    ('E1/test.csv',  df_test_v,  'test'),
]
todo_ok = True
for archivo, df_v, clave in filas:
    hash_meta = metadata[f'hash_md5_{clave}']
    hash_disk = hashes_v[clave]
    estado    = 'OK' if hash_meta == hash_disk else 'ERROR'
    if estado == 'ERROR':
        todo_ok = False
    print(f'{archivo:<20} {len(df_v):>10} {hash_disk:<34} {estado:>6}')

print()
if todo_ok:
    print('Dataset E1 verificado y listo para fine-tuning.')
else:
    raise RuntimeError('Se detectaron discrepancias en los hashes MD5.')

Archivo               Registros Hash MD5                           Estado
-------------------------------------------------------------------------
E1/train.csv                957 d5949dfe6b3c2f42110b6865895fbebd       OK
E1/val.csv                   97 2d22860c2f6cbfeccca34a75f1d356ff       OK
E1/test.csv                  97 06697b127fd01b181ff5e50d87a5a2d3       OK

Dataset E1 verificado y listo para fine-tuning.
